# 05 — Phase 0.5: Linear extrapolation baseline (issue #3)

Fit a straight line to the last `n_recent` observed heel rows of `(MD, TVT_input)`
and project that slope across the toe eval zone. Compare against the **carry-forward
floor (11.53 ft RMSE)** from `notebooks/00` / `docs/decisions.md`.

Hypothesis (issue #3): wells continue their heel-exit dip into the toe, so a
non-zero slope should beat the flat carry-forward on drift-y wells.

This notebook mirrors the eval harness of `notebooks/10_dtw_alignment.ipynb`
**exactly** — same RNG seed (0), same 10 wells, same eval mask, same `rmse` helper —
so the comparison is apples-to-apples.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Bounded upward walk to the repo root (issue #9), then put src on path.
repo = Path.cwd().resolve()
for _ in range(40):
    if (repo / 'pyproject.toml').exists():
        break
    if repo.parent == repo:
        raise RuntimeError(f'pyproject.toml not found walking up from {Path.cwd()}')
    repo = repo.parent
else:
    raise RuntimeError('pyproject.toml not found within 40 levels')
sys.path.insert(0, str(repo / 'src'))

from rogii.features.correlation import rmse
from rogii.models.linear_extrap import predict_carry_forward, predict_linear_extrap

TRAIN = repo / 'data' / 'raw' / 'train'
print('Repo:', repo)
print('Train wells:', len(list(TRAIN.glob('*__horizontal_well.csv'))))

In [ ]:
# Same 10-well sample as notebooks/10_dtw_alignment.ipynb (RNG seed = 0).
wells = sorted({p.name.split('__')[0] for p in TRAIN.glob('*__horizontal_well.csv')})
rng = np.random.default_rng(0)
sample_wells = list(rng.choice(wells, size=10, replace=False))
print('Wells sampled:', sample_wells)

def load(well: str) -> pd.DataFrame:
    return pd.read_csv(TRAIN / f'{well}__horizontal_well.csv')

## Sweep `n_recent ∈ {50, 100, 200, 500, ALL}`

`None` fits the line on **all** observed heel rows. The metric is eval-zone RMSE
against the train-only ground-truth `TVT` column.

In [ ]:
N_RECENT_SWEEP = [50, 100, 200, 500, None]   # None = ALL heel rows
CARRY_FORWARD_FLOOR = 11.53                    # docs/decisions.md 2026-05-05
label = lambda n: 'ALL' if n is None else str(n)

rows = []
for well in sample_wells:
    h = load(well)
    eval_mask = h['TVT_input'].isna().to_numpy()
    y_true = h['TVT'].to_numpy(float)
    cf = predict_carry_forward(h)
    rec = {
        'well': well,
        'eval_rows': int(eval_mask.sum()),
        'heel_rows': int((~eval_mask).sum()),
        'rmse_cf': rmse(y_true[eval_mask], cf[eval_mask]),
    }
    for n in N_RECENT_SWEEP:
        lin = predict_linear_extrap(h, n_recent=n)
        rec[f'rmse_lin_{label(n)}'] = rmse(y_true[eval_mask], lin[eval_mask])
    rows.append(rec)

df = pd.DataFrame(rows)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
print('Per-well eval-zone RMSE (ft):')
print(df.round(2).to_string(index=False))

In [ ]:
# Aggregate + wins-vs-floor per window.
print(f"{'n_recent':>10} | {'agg RMSE':>9} | {'vs floor':>9} | wins/10")
print('-' * 48)
print(f"{'carry-fwd':>10} | {df['rmse_cf'].mean():9.2f} | {'(floor)':>9} | {'-':>5}")
for n in N_RECENT_SWEEP:
    col = f'rmse_lin_{label(n)}'
    agg = df[col].mean()
    wins = int((df[col] < df['rmse_cf']).sum())
    print(f'{label(n):>10} | {agg:9.2f} | {agg - CARRY_FORWARD_FLOOR:+9.2f} | {wins:>5}/10')
print()
print(f'Reference floor (decisions.md): {CARRY_FORWARD_FLOOR} ft')
print(f"This run carry-forward agg:      {df['rmse_cf'].mean():.2f} ft")

# Plan decision rule: canonical window = n_recent=200, adopt if wins >= 6/10.
w200 = int((df['rmse_lin_200'] < df['rmse_cf']).sum())
print(f"\nDecision window n_recent=200: agg={df['rmse_lin_200'].mean():.2f} ft, wins {w200}/10 (adopt if >=6/10).")

## Result

Carry-forward reproduces the **11.53 ft** floor exactly. Linear extrapolation
**loses at every window** (best n_recent=200 ≈ 66 ft, winning on only 1/10 wells;
the ALL-heel fit blows up because it includes the steep heel-build section).

**Decision (plan rule):** wins < 6/10 → **keep carry-forward as the Phase-0 floor.**
Most wells stay near their heel-exit TVT through the toe, so a projected slope
over a ~5000-ft toe diverges from the (near-flat) truth even with the clip guard.
The clip guard works (caps the blow-up) but cannot rescue a wrong slope.
See `docs/decisions.md` for the full write-up incl. the n=10 sample-size caveat.

Negative result — recorded as the informative baseline; Phase 1 v2 still targets 11.53 ft.